# Fine-tune logo/field detector (YOLOv8) on Colab

Continues training from your existing `best_colab2.pt` on the merged dataset (628 images), instead of starting over from generic COCO weights - converges faster since the model already knows the 9-class scheme.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU -> Save.

In [ ]:
# 1. Confirm you actually got a GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics

In [ ]:
# 2b. Mount Google Drive (upload final_dataset.zip to your Drive FIRST,
# via drive.google.com's web uploader - not through this notebook)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Copy final_dataset.zip from Drive (fast, Google-internal network) and
# unzip it into Colab's local disk (faster to train against than reading
# directly off the Drive mount during training).
#
# Update this path if you uploaded the zip into a subfolder of My Drive.
import zipfile, os

DRIVE_ZIP_PATH = "/content/drive/MyDrive/final_dataset.zip"

with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as z:
    z.extractall('/content/')

print(os.listdir('/content/final_dataset'))

In [ ]:
# 3b. Upload your existing best_colab2.pt (the model to fine-tune from)
from google.colab import files
weights_uploaded = files.upload()  # pick models/best_colab2.pt from your PC
weights_name = next(iter(weights_uploaded))
print("Starting from:", weights_name)

In [ ]:
# 4. Fix the hardcoded Windows path inside logo_dataset.yaml
yaml_path = "/content/final_dataset/logo_dataset.yaml"
content = open(yaml_path).read()
content = content.replace("C:/Users/xian.xin/MarkLogic/final_dataset", "/content/final_dataset")
open(yaml_path, "w").write(content)
print(open(yaml_path).read())

In [ ]:
# 6. Fine-tune from your existing best_colab2.pt (not a fresh COCO checkpoint)
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else "cpu"
print("Training on:", "GPU" if device == 0 else "CPU")

model = YOLO(weights_name)  # continues from best_colab2.pt's learned weights

model.train(
    name="finetune-colab",
    exist_ok=True,
    data=yaml_path,
    epochs=100,         # fine-tuning from a decent checkpoint converges faster than
                        # training from scratch - patience=30 will stop earlier if it
                        # plateaus sooner than that anyway
    imgsz=640,
    batch=16,           # Colab's T4 has far more VRAM than 4GB
    device=device,
    workers=4,
    amp=True,
    cache=True,
    patience=30,
    val=True,
    warmup_epochs=3,    # shorter warmup - the model isn't starting from random weights
    optimizer="auto",
    augment=True,
    flipud=0.1,
    fliplr=0.5,
    mosaic=1.0,
    degrees=5.0,
    scale=0.3,
)

In [ ]:
# 7. Download everything - not just the weights.
# Ultralytics already writes the same training-progress artifacts
# train_logo.py produces locally (results.csv, results.png, confusion
# matrix, PR/F1 curves, sample val batch predictions) into
# runs/detect/finetune-colab/ - they just weren't being downloaded before.
import shutil
from google.colab import files

run_dir = "runs/detect/finetune-colab"

# Weights on their own, for quick direct access
files.download(f"{run_dir}/weights/best.pt")

# Everything else (curves, confusion matrix, val prediction samples, args.yaml)
shutil.make_archive("finetune-colab-results", "zip", run_dir)
files.download("finetune-colab-results.zip")